# Sequence Annotation Objects (Biopython) — Full Guided Lesson

## Key concepts (before coding)
- **Seq**: biological sequence object.
- **SeqRecord**: sequence + metadata (`id`, `name`, `description`, `annotations`, `features`, `dbxrefs`, `letter_annotations`).
- **SeqFeature**: biologically meaningful region (gene, CDS, etc.).
- **Location**: feature interval on parent sequence (`start:end`, strand).
- **Position**: single boundary coordinate; exact or fuzzy.
- **Per-letter annotation**: one value per base/residue (e.g., FASTQ quality), must match sequence length.
- **Python indexing**: 0-based, end-exclusive.

## Section plan
1. Setup and files.
2. Build SeqRecord from scratch.
3. Read FASTA and GenBank.
4. Understand features/locations/fuzzy positions.
5. Query coordinates with `in`.
6. Extract feature sequence safely.
7. Compare, format, slice, add, shift, reverse-complement SeqRecord objects.

In [2]:
# Line 1: Import filesystem path helper.
from pathlib import Path
# Line 2: Import downloader for tutorial files.
from urllib.request import urlretrieve
# Line 3: Import Biopython and SeqIO parser.
import Bio
from Bio import SeqIO

# Line 4: Define data directory inside your project.
data_dir = Path(r"C:/Users/hp/OneDrive/Desktop/projects/bio/data")
# Line 5: Create data directory if missing.
data_dir.mkdir(parents=True, exist_ok=True)

# Line 6: Define required tutorial files and source URLs.
files_to_fetch = {
    "NC_005816.fna": "https://raw.githubusercontent.com/biopython/biopython/master/Tests/GenBank/NC_005816.fna",
    "NC_005816.gb": "https://raw.githubusercontent.com/biopython/biopython/master/Tests/GenBank/NC_005816.gb",
    "example.fastq": "https://raw.githubusercontent.com/biopython/biopython/master/Tests/Quality/example.fastq",
}

# Line 7: Download files only when absent.
for name, url in files_to_fetch.items():
    target = data_dir / name
    if not target.exists():
        try:
            urlretrieve(url, target)
            print("Downloaded:", name)
        except Exception as exc:
            print("Download failed:", name, "->", exc)

# Line 8: Print environment and file readiness.
print("Biopython version:", Bio.__version__)
for name in files_to_fetch:
    print("Ready path:", data_dir / name)

Biopython version: 1.86
Ready path: C:\Users\hp\OneDrive\Desktop\projects\bio\data\NC_005816.fna
Ready path: C:\Users\hp\OneDrive\Desktop\projects\bio\data\NC_005816.gb
Ready path: C:\Users\hp\OneDrive\Desktop\projects\bio\data\example.fastq


## 1) SeqRecord from scratch

A minimal SeqRecord needs only a `Seq`, but practical usage sets:
- id/name/description
- annotations
- per-letter annotations
- dbxrefs
- features

In [3]:
# Line 1: Import sequence and record classes.
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
# Line 2: Import feature classes.
from Bio.SeqFeature import SeqFeature, SimpleLocation # this means we are importing the SeqFeature and SimpleLocation classes from the Bio.SeqFeature module. The SeqFeature class is used to represent a feature on a sequence, such as a gene or a regulatory element, while the SimpleLocation class is used to specify the location of that feature on the sequence, including its start and end positions and strand orientation.

# Line 3: Create a DNA sequence.
simple_seq = Seq("GATC")
# Line 4: Wrap it into a SeqRecord.
simple_seq_r = SeqRecord(simple_seq)

# Line 5: Show default identity placeholders.
print("Default id:", simple_seq_r.id)
print("Default name:", simple_seq_r.name)
print("Default description:", simple_seq_r.description)

# Line 6: Set identity metadata.
simple_seq_r.id = "AC12345"
simple_seq_r.name = "DemoClone"
simple_seq_r.description = "Made up sequence I wish I could write a paper about"

# Line 7: Add general annotation dictionary entries.
simple_seq_r.annotations["evidence"] = "None. I just made it up." # the annotations attribute of the simple_seq_r object is a dictionary that can be used to store additional information about the sequence record. In this case, we are adding a key-value pair to the annotations dictionary, where the key is "evidence" and the value is "None. I just made it up." This means that there is no evidence supporting the existence of this sequence, and it was created purely for demonstration purposes.

# Line 8: Add per-letter quality values (length must equal sequence length).
simple_seq_r.letter_annotations["phred_quality"] = [40, 40, 38, 30] # the letter_annotations attribute of the simple_seq_r object is a dictionary that can be used to store per-letter annotations, such as quality scores for each base in the sequence. In this case, we are adding a key-value pair to the letter_annotations dictionary, where the key is "phred_quality" and the value is a list of integers representing the quality scores for each base in the sequence. The length of this list must match the length of the sequence itself, which is 4 in this case (G, A, T, C). The quality scores are typically represented as Phred scores, which indicate the confidence level of each base call in sequencing data. Higher values indicate higher confidence in the accuracy of the base call.

# Line 9: Add an external database cross-reference.
simple_seq_r.dbxrefs.append("Project:Demo123") # the dbxrefs attribute of the simple_seq_r object is a list that can be used to store external database cross-references related to this sequence record. In this case, we are appending a string "Project:Demo123" to the dbxrefs list, which indicates that this sequence record is associated with a project named "Demo123". This allows for easy linking and referencing of this sequence record in external databases or projects.

# Line 10: Add a feature object (gene on interval [1:3], + strand).
demo_feature = SeqFeature(
    SimpleLocation(1, 3, strand=1),
    type="gene",
    qualifiers={"gene": ["demoA"]},
) # this block of code is creating a feature object called demo_feature using the SeqFeature class. The feature is defined with a location specified by SimpleLocation(1, 3, strand=1), which means it starts at position 1, ends at position 3, and is on the positive strand. The type of the feature is set to "gene", and it has a qualifier dictionary with a key "gene" and a value of a list containing the string "demoA". This means that this feature represents a gene named "demoA" located on the positive strand between positions 1 and 3 of the sequence.
simple_seq_r.features.append(demo_feature)

# Line 11: Display the complete object state.
print("this is the complete object:",simple_seq_r)
print("seq:", simple_seq_r.seq)
print("annotations:", simple_seq_r.annotations)
print("letter_annotations:", simple_seq_r.letter_annotations)
print("dbxrefs:", simple_seq_r.dbxrefs) #
print("features count:", len(simple_seq_r.features))

Default id: <unknown id>
Default name: <unknown name>
Default description: <unknown description>
this is the complete object: ID: AC12345
Name: DemoClone
Description: Made up sequence I wish I could write a paper about
Database cross-references: Project:Demo123
Number of features: 1
/evidence=None. I just made it up.
Per letter annotation for: phred_quality
Seq('GATC')
seq: GATC
annotations: {'evidence': 'None. I just made it up.'}
letter_annotations: {'phred_quality': [40, 40, 38, 30]}
dbxrefs: ['Project:Demo123']
features count: 1


## 2) Read a FASTA record

FASTA usually gives:
- id and name from first token after `>`
- description from full title line
- little/no structured annotation

In [4]:
# Line 1: Read one FASTA record.
record_fasta = SeqIO.read(data_dir / "NC_005816.fna", "fasta")

# Line 2: Print compact summary.
print(record_fasta)

# Line 3: Inspect identity fields.
print("id:", record_fasta.id)
print("name:", record_fasta.name)
print("description:", record_fasta.description)

# Line 4: FASTA generally lacks rich metadata.
print("dbxrefs:", record_fasta.dbxrefs)
print("annotations:", record_fasta.annotations)
print("letter_annotations:", record_fasta.letter_annotations)
print("features:", record_fasta.features)

ID: gi|45478711|ref|NC_005816.1|
Name: gi|45478711|ref|NC_005816.1|
Description: gi|45478711|ref|NC_005816.1| Yersinia pestis biovar Microtus str. 91001 plasmid pPCP1, complete sequence
Number of features: 0
Seq('TGTAACGAACGGTGCAATAGTGATCCACACCCAACGCCTGAAATCAGATCCAGG...CTG')
id: gi|45478711|ref|NC_005816.1|
name: gi|45478711|ref|NC_005816.1|
description: gi|45478711|ref|NC_005816.1| Yersinia pestis biovar Microtus str. 91001 plasmid pPCP1, complete sequence
dbxrefs: []
annotations: {}
letter_annotations: {}
features: []


## 3) Read a GenBank record

GenBank is richer:
- annotations dictionary populated
- feature list populated
- dbxrefs may be present

In [5]:
# Line 1: Read one GenBank record.
record_gb = SeqIO.read(data_dir / "NC_005816.gb", "genbank")

# Line 2: Print summary.
print(record_gb)

# Line 3: Show identity fields.
print("id:", record_gb.id)
print("name:", record_gb.name)
print("description:", record_gb.description)

# Line 4: Show rich metadata.
print("len(annotations):", len(record_gb.annotations))
print("source annotation:", record_gb.annotations.get("source"))
print("dbxrefs:", record_gb.dbxrefs)
print("len(features):", len(record_gb.features))
print("letter_annotations:", record_gb.letter_annotations)

ID: NC_005816.1
Name: NC_005816
Description: Yersinia pestis biovar Microtus str. 91001 plasmid pPCP1, complete sequence
Database cross-references: Project:58037
Number of features: 41
/molecule_type=DNA
/topology=circular
/data_file_division=BCT
/date=21-JUL-2008
/accessions=['NC_005816']
/sequence_version=1
/gi=45478711
/keywords=['']
/source=Yersinia pestis biovar Microtus str. 91001
/organism=Yersinia pestis biovar Microtus str. 91001
/taxonomy=['Bacteria', 'Proteobacteria', 'Gammaproteobacteria', 'Enterobacteriales', 'Enterobacteriaceae', 'Yersinia']
/references=[Reference(title='Genetics of metabolic variations between Yersinia pestis biovars and the proposal of a new biovar, microtus', ...), Reference(title='Complete genome sequence of Yersinia pestis strain 91001, an isolate avirulent to humans', ...), Reference(title='Direct Submission', ...), Reference(title='Direct Submission', ...)]
/comment=PROVISIONAL REFSEQ: This record has not yet been subject to final
NCBI review. The 

## 4) Inspect feature structure (`type`, `location`, `strand`, `qualifiers`)

In [6]:
# Line 1: Pick tutorial feature indices when available.
indices = [20, 21] if len(record_gb.features) > 21 else [0]

# Line 2: Inspect each selected feature.
for idx in indices:
    feature = record_gb.features[idx]
    print("\nFeature index:", idx)
    print("type:", feature.type)
    print("location:", feature.location)
    print("strand:", feature.location.strand)
    print("qualifier keys:", sorted(feature.qualifiers.keys()))
    print("db_xref:", feature.qualifiers.get("db_xref"))


Feature index: 20
type: gene
location: [4342:4780](+)
strand: 1
qualifier keys: ['db_xref', 'gene', 'locus_tag']
db_xref: ['GeneID:2767712']

Feature index: 21
type: CDS
location: [4342:4780](+)
strand: 1
qualifier keys: ['codon_start', 'db_xref', 'gene', 'locus_tag', 'note', 'product', 'protein_id', 'transl_table', 'translation']
db_xref: ['GI:45478716', 'GeneID:2767712']


## 5) Positions and fuzzy locations

Key idea:
- position = one boundary
- location = interval between boundaries

Fuzzy examples include `AfterPosition`, `BeforePosition`, and `BetweenPosition`.

In [7]:
# Line 1: Import SeqFeature namespace for position/location classes.
from Bio import SeqFeature as SF

# Line 2: Create fuzzy start (>5).
start_pos = SF.AfterPosition(5)

# Line 3: Create fuzzy end with compatibility fallback.
if hasattr(SF, "BetweenPosition"):
    end_pos = SF.BetweenPosition(9, left=8, right=9)
else:
    end_pos = SF.WithinPosition(9, left=8, right=9)

# Line 4: Build fuzzy location.
my_location = SF.SimpleLocation(start_pos, end_pos)

# Line 5: Print fuzzy location and integer conversions.
print("my_location:", my_location)
print("start object:", my_location.start, "| int:", int(my_location.start))
print("end object:", my_location.end, "| int:", int(my_location.end))

# Line 6: Build exact location from integers.
exact_location = SF.SimpleLocation(5, 9)
print("exact_location:", exact_location)
print("exact start type:", type(exact_location.start).__name__)
print("exact end type:", type(exact_location.end).__name__)

my_location: [>5:(8^9)]
start object: >5 | int: 5
end object: (8^9) | int: 9
exact_location: [5:9]
exact start type: ExactPosition
exact end type: ExactPosition


## 6) Coordinate membership testing with `in`

Use case: “Which features contain this SNP coordinate?”

In [8]:
# Line 1: Define a SNP coordinate in Python indexing.
my_snp = 4350

# Line 2: Iterate over features and test membership.
hits = []
for feature in record_gb.features:
    if my_snp in feature:
        hits.append((feature.type, feature.qualifiers.get("db_xref")))

# Line 3: Print all overlapping features.
print("SNP coordinate:", my_snp)
for feature_type, refs in hits:
    print(feature_type, refs)

SNP coordinate: 4350
source ['taxon:229193']
gene ['GeneID:2767712']
CDS ['GI:45478716', 'GeneID:2767712']


## 7) Extract sequence described by a feature

Preferred method: `feature.extract(parent_seq)` because it handles strand and complex locations.

In [9]:
# Line 1: Import compact classes for toy demonstration.
from Bio.Seq import Seq
from Bio.SeqFeature import SeqFeature, SimpleLocation

# Line 2: Define a short parent sequence.
seq = Seq("ACCGAGACGGCAAAGGCTAGCATAGGTATGAGACTTCCTTCCTGCCAGTGCTGAGGAACTGGGAGCCTAC")

# Line 3: Define reverse-strand feature over [5:18].
feature = SeqFeature(SimpleLocation(5, 18, strand=-1), type="gene")

# Line 4: Manual extraction (slice then reverse complement).
manual = seq[feature.location.start : feature.location.end].reverse_complement()

# Line 5: Built-in extraction.
auto = feature.extract(seq)

# Line 6: Validate equivalence and lengths.
print("manual:", manual)
print("auto  :", auto)
print("same? :", manual == auto)
print("len(auto):", len(auto))
print("len(feature):", len(feature))
print("len(feature.location):", len(feature.location))

manual: AGCCTTTGCCGTC
auto  : AGCCTTTGCCGTC
same? : True
len(auto): 13
len(feature): 13
len(feature.location): 13


## 8) SeqRecord comparison

Direct `record1 == record2` is deliberately not implemented.
Compare explicit attributes (`id`, `seq`, etc.) instead.

In [10]:
# Line 1: Import minimal classes.
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

# Line 2: Create two records with same visible content.
record1 = SeqRecord(Seq("ACGT"), id="test")
record2 = SeqRecord(Seq("ACGT"), id="test")

# Line 3: Show modern comparison behavior.
try:
    print(record1 == record2)
except NotImplementedError as exc:
    print("Expected NotImplementedError:", exc)

# Line 4: Compare explicit attributes safely.
print("id equal:", record1.id == record2.id)
print("seq equal:", record1.seq == record2.seq)

Expected NotImplementedError: SeqRecord comparison is deliberately not implemented. Explicitly compare the attributes of interest.
id equal: True
seq equal: True


## 9) `format()` method

`record.format("fasta")` returns a text string in the chosen output format.

In [11]:
# Line 1: Import classes for formatting example.
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

# Line 2: Build a protein record.
record_fmt = SeqRecord(
    Seq(
        "MMYQQGCFAGGTVLRLAKDLAENNRGARVLVVCSEITAVTFRGPSETHLDSMVGQALFGD"
        "GAGAVIVGSDPDLSVERPLYELVWTGATLLPDSEGAIDGHLREVGLTFHLLKDVPGLISK"
        "NIEKSLKEAFTPLGISDWNSTFWIAHPGGPAILDQVEAKLGLKEEKMRATREVLSEYGNM"
        "SSAC"
    ),
    id="gi|14150838|gb|AAK54648.1|AF376133_1",
    description="chalcone synthase [Cucumis sativus]",
)

# Line 3: Convert record to FASTA text.
fasta_text = record_fmt.format("fasta")

# Line 4: Print result.
print(fasta_text)

>gi|14150838|gb|AAK54648.1|AF376133_1 chalcone synthase [Cucumis sativus]
MMYQQGCFAGGTVLRLAKDLAENNRGARVLVVCSEITAVTFRGPSETHLDSMVGQALFGD
GAGAVIVGSDPDLSVERPLYELVWTGATLLPDSEGAIDGHLREVGLTFHLLKDVPGLISK
NIEKSLKEAFTPLGISDWNSTFWIAHPGGPAILDQVEAKLGLKEEKMRATREVLSEYGNM
SSAC



## 10) Slice a SeqRecord

When slicing:
- sequence is sliced
- per-letter annotations are sliced
- fully contained features are kept and re-based
- most global annotations/dbxrefs are dropped for safety

In [12]:
# Line 1: Slice a region that includes pim gene/CDS.
sub_record = record_gb[4300:4800]

# Line 2: Print size and feature count.
print("len(sub_record):", len(sub_record))
print("len(sub_record.features):", len(sub_record.features))

# Line 3: Show adjusted feature locations.
for f in sub_record.features:
    print(f.type, f.location)

# Line 4: Show preserved vs dropped metadata.
print("annotations:", sub_record.annotations)
print("dbxrefs:", sub_record.dbxrefs)

# Line 5: Fix context-specific metadata for partial fragment.
sub_record.annotations["topology"] = "linear"
sub_record.description = "Yersinia pestis biovar Microtus str. 91001 plasmid pPCP1, partial"

# Line 6: Preview GenBank text.
print(sub_record.format("genbank")[:240] + "...")

len(sub_record): 500
len(sub_record.features): 2
gene [42:480](+)
CDS [42:480](+)
annotations: {'molecule_type': 'DNA'}
dbxrefs: []
LOCUS       NC_005816                500 bp    DNA     linear   UNK 01-JAN-1980
DEFINITION  Yersinia pestis biovar Microtus str. 91001 plasmid pPCP1, partial.
ACCESSION   NC_005816
VERSION     NC_005816.1
KEYWORDS    .
SOURCE      .
  ORGAN...


## 11) Add SeqRecords (FASTQ edit pattern)

`edited = left + right` keeps compatible per-letter annotations aligned.

In [13]:
# Line 1: Read first FASTQ record.
record_fastq = next(SeqIO.parse(data_dir / "example.fastq", "fastq"))

# Line 2: Inspect original sequence and qualities.
print("Original length:", len(record_fastq))
print("Original seq:", record_fastq.seq)
print("Original qualities:", record_fastq.letter_annotations["phred_quality"])

# Line 3: Remove one suspected extra base by slicing around index 20.
left = record_fastq[:20]
right = record_fastq[21:]

# Line 4: Concatenate slices to create edited record.
edited = left + right

# Line 5: Validate output sequence and qualities.
print("Edited length:", len(edited))
print("Edited seq:", edited.seq)
print("Edited qualities:", edited.letter_annotations["phred_quality"])

Original length: 25
Original seq: CCCTTCTTGTCTTCAGCGTTTCTCC
Original qualities: [26, 26, 18, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 22, 26, 26, 26, 26, 26, 26, 26, 23, 23]
Edited length: 24
Edited seq: CCCTTCTTGTCTTCAGCGTTCTCC
Edited qualities: [26, 26, 18, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 26, 22, 26, 26, 26, 26, 26, 26, 23, 23]


## 12) Shift circular origin by record addition

Pattern:
`shifted = record[cut:] + record[:cut]`

In [14]:
# Line 1: Shift origin at coordinate 2000.
shifted = record_gb[2000:] + record_gb[:2000]

# Line 2: Compare size/features before and after.
print("Original length:", len(record_gb), "| Shifted length:", len(shifted))
print("Original features:", len(record_gb.features), "| Shifted features:", len(shifted.features))

# Line 3: Show cautious metadata loss after slicing+adding.
print("shifted.dbxrefs before:", shifted.dbxrefs)
print("shifted.annotations keys before:", list(shifted.annotations.keys()))

# Line 4: Restore metadata explicitly if biologically valid.
shifted.dbxrefs = record_gb.dbxrefs[:]
shifted.annotations = record_gb.annotations.copy()

# Line 5: Confirm restored metadata.
print("shifted.dbxrefs after:", shifted.dbxrefs)
print("shifted.annotations keys after:", list(shifted.annotations.keys()))

Original length: 9609 | Shifted length: 9609
Original features: 41 | Shifted features: 40
shifted.dbxrefs before: []
shifted.annotations keys before: ['molecule_type']
shifted.dbxrefs after: ['Project:58037']
shifted.annotations keys after: ['molecule_type', 'topology', 'data_file_division', 'date', 'accessions', 'sequence_version', 'gi', 'keywords', 'source', 'organism', 'taxonomy', 'references', 'comment']


## 13) Reverse-complement a SeqRecord

`reverse_complement()`:
- reverse-complements sequence
- remaps feature locations/strands
- reverses per-letter annotations
- drops many metadata fields by default unless requested

In [15]:
# Line 1: Alias original record.
rec = record_gb

# Line 2: Reverse-complement and set new id.
rc = rec.reverse_complement(id="TESTING")

# Line 3: Compare summary statistics.
print("Original:", rec.id, len(rec), len(rec.features), len(rec.dbxrefs), len(rec.annotations))
print("RC      :", rc.id, len(rc), len(rc.features), len(rc.dbxrefs), len(rc.annotations))

Original: NC_005816.1 9609 41 1 13
RC      : TESTING 9609 41 0 0


## Final recap

You now have a full, sectioned, teaching-first notebook with:
- definitions before implementation
- line-commented code
- practical examples throughout